# Setup notebook environment

This notebook resolves the repo root in the Databricks Git folder checkout,
defines shared path constants, adds `SRC_ROOT` to `sys.path` so package imports work across notebooks,
and selects the `derm_showcase_project` catalog (`USE CATALOG`) -- session-scoped, not persistent
across notebook runs, so every notebook re-issues it here rather than relying on another notebook
having run earlier in the same session. Tolerates the catalog not existing yet (fresh workspace,
`00_setup_storage_and_shared_tables.ipynb` never having run) -- prints a warning and continues rather
than failing, since `00` is the one place that actually creates the catalog, not this notebook.

In [ ]:
import sys
from pathlib import Path

workspace_notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
candidate_bases = [Path.cwd(), Path(f"/Workspace{workspace_notebook_path}").parent]

REPO_ROOT = None
for base_path in candidate_bases:
    for candidate in [base_path, *base_path.parents]:
        possible_src = candidate / "src"
        possible_config = candidate / "config"
        if possible_src.exists() and possible_config.exists():
            REPO_ROOT = candidate
            break
    if REPO_ROOT is not None:
        break

if REPO_ROOT is None:
    raise RuntimeError("Could not locate the repo root. Open the notebook from the Databricks Git folder checkout.")

SRC_ROOT = REPO_ROOT / "src"
CONFIG_ROOT = REPO_ROOT / "config"

if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

print(f"Using repo root: {REPO_ROOT}")
print(f"Using src root: {SRC_ROOT}")
print(f"Using config root: {CONFIG_ROOT}")

CATALOG_NAME = "derm_showcase_project"
try:
    spark.sql(f"USE CATALOG {CATALOG_NAME}")
    print(f"Using catalog: {CATALOG_NAME}")
except Exception as error:
    print(
        f"Could not select catalog {CATALOG_NAME!r} -- it likely does not exist yet. "
        "Run 00_setup_storage_and_shared_tables.ipynb first to create it (fresh workspace only). "
        f"Original error: {error}"
    )
